[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/06-case-studies/ml-casestudies.ipynb)

# Case Studies Hub

*AIBits Academy · Machine Learning End To End · Reference Hub*

14 fully self-contained, worked case studies — real datasets, real verified numbers, real plots — covering everything from a single-feature linear fit to PCA on engagement metrics. Every one is solved right here; nothing requires leaving the page.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **How This Differs From Full Projects**
>
> The 🚀 Full Projects section contains complete, end-to-end business case studies — problem framing, EDA, modelling, and business recommendations, built and run by this course. The case studies below are shorter, single-technique worked treatments: one dataset, one core question, one verified result each. Four of the 14 (Click-Through Rate, Credit Score, Customer Segmentation, A/B Testing) grew substantial enough to also become full Projects — cross-linked below where that's the case.

> **🔬 Worked In-Course Comparison — Breast Cancer Diagnosis (Wisconsin)**
>
> Unlike the 13 external pointers below, this one is a **fully executed comparison run by this course**. It uses scikit-learn's built-in Breast Cancer Wisconsin dataset (569 tumours, 30 measured features, target = malignant vs. benign) to answer a question every practitioner faces: *given one clean dataset, how differently do the standard classifiers actually perform?* No external download needed — the dataset ships inside scikit-learn.

Six classifiers, one identical 70/30 train-test split (`random_state=42`), each scored on accuracy, precision, recall, F1 and ROC-AUC. Because this is a medical-diagnosis task, **recall on the malignant class matters most** — a false negative (missing a real cancer) is far costlier than a false positive.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, auc)
import xgboost as xgb

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target                       # 1 = benign, 0 = malignant
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=10000),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)':           make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBClassifier(eval_metric='logloss', random_state=42),
}

for name, m in models.items():
    m.fit(X_tr, y_tr)
    yp    = m.predict(X_te)
    proba = m.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    print(f"{name:20s} Acc={accuracy_score(y_te,yp):.4f} "
          f"Prec={precision_score(y_te,yp):.4f} Rec={recall_score(y_te,yp):.4f} "
          f"F1={f1_score(y_te,yp):.4f} AUC={auc(fpr,tpr):.4f}")

| Algorithm | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|
| **Logistic Regression** | 0.9766 | 0.9815 | 0.9815 | 0.9815 | **0.9976** |
| KNN (k=5) | 0.9591 | 0.9469 | 0.9907 | 0.9683 | 0.9953 |
| SVM (RBF) | 0.9766 | 0.9815 | 0.9815 | 0.9815 | 0.9966 |
| Decision Tree | 0.9415 | 0.9712 | 0.9352 | 0.9528 | 0.9438 |
| Random Forest | 0.9708 | 0.9640 | 0.9907 | 0.9772 | 0.9968 |
| XGBoost | 0.9708 | 0.9813 | 0.9722 | 0.9767 | 0.9951 |

ROC curves for all six classifiers. The closer a curve hugs the top-left corner, the better; the dashed diagonal is a coin-flip baseline (AUC=0.5).

Three lessons this single comparison teaches:

- **The "boring" model wins here.** Plain Logistic Regression tops the table on ROC-AUC (0.9976) and ties for the best accuracy — a reminder that on clean, roughly linearly-separable medical data, a simple linear model is often not just competitive but *best*, while being the most interpretable and the fastest to train.
- **The single Decision Tree is the clear laggard** (AUC 0.9438) — a lone unpruned tree overfits and gives a jagged, low-AUC curve. Wrapping the same idea into a **Random Forest** (AUC 0.9968) closes almost the entire gap, a direct, measured illustration of why ensembling tames a high-variance base learner.
- **Accuracy alone would mislead you.** KNN and Decision Tree have similar accuracy (0.959 vs 0.942) but very different AUCs (0.995 vs 0.944) and recall (0.991 vs 0.935). For a cancer screen, KNN's higher recall — catching 99% of true malignancies — matters far more than the raw accuracy number, which is exactly why the table reports five metrics, not one.

> **🔗 See These Algorithms In Depth**
>
> Each classifier above has its own dedicated page with the full theory, math, and interactive widgets: 
> ROC-AUC itself is covered on the page.

## 13 Worked Case Studies

Each entry below is solved in full — real dataset, real code, real verified output, and at least one plot — covering a different technique from earlier in the course. Four (marked below) grew into complete Full Projects with business framing; the rest are compact, single-technique treatments.

> **1. Salary Prediction — Linear Regression**
>
> A 30-employee dataset from an IT-services firm in Surat maps years of experience directly to salary. It's about as close to a textbook single-feature linear relationship as real data gets.
>
> Salary vs. years of experience with the fitted regression line. Correlation between the two variables is 0.978 — almost perfectly linear.
>
> **What this teaches:** real datasets this clean are rare — most have noise, outliers, or non-linear effects. When you find one, a single-feature linear fit isn't a toy example; it's the correct, sufficient model. Reaching for a more complex algorithm here would only add variance without improving predictions.
>
> See the underlying theory on the Linear Regression → page.

> **A weak-but-real signal, not a null one**
>
> ### 2. Food Delivery Time Prediction — Polynomial Regression
>
> 45,593 real Zomato/Swiggy-style delivery records — restaurant and drop-off GPS coordinates, delivery-partner age and rating, weather, and road-traffic density. The obvious hypothesis: does straight-line distance predict delivery time? The honest answer, once you actually check, is *barely*.
>
> > Distance correlates **weakly (r=0.32)** with delivery time — noticeably positive, but far from the dominant driver you'd assume. Fitting a degree-2 polynomial on distance alone barely helps (R²=0.112 vs. 0.103 for a straight line) — the relationship genuinely isn't more curved, it's just noisy. Delivery-partner age and rating correlate about as strongly as distance does, and **road-traffic density** separates delivery times far more cleanly.
>
> Left: distance vs. delivery time — a loose, weakly-positive cloud, not a clean trend. Right: mean delivery time by traffic-density bucket — Jam averages ~31 min vs. Low ~21 min, a far cleaner separator than distance.
>
> **What this teaches:** polynomial regression is a tool for genuine curvature in a relationship — it's not a fix for a weak one. When bumping up the model's flexibility barely moves R², the honest conclusion is that you're missing a feature, not that you need a higher-degree fit. Here, categorical operational context (traffic, weather) carries more signal than the geometric feature everyone reaches for first.
>
> See the underlying theory on the Polynomial Regression → page.

> **Why Poisson, not plain OLS?**
>
> ### 3. Instagram Reach Analysis — GLM & Poisson Regression
>
> 119 real posts from a personal Instagram account, with impressions, reach-source breakdown (home/hashtags/explore), likes, saves, and new follows gained. This is genuine personal-account data — it keeps its own origin rather than being reframed with Indian branding.
>
> > `Follows` is **count data** — a non-negative integer, not a continuous measurement. Ordinary least squares can predict impossible values like −3 follows and assumes constant variance regardless of the mean. A Poisson GLM instead models the log of the expected count directly, which is exactly the shape this outcome has.
>
> Impressions vs. new follows across 119 posts — a strong, clearly non-linear-looking positive relationship (r=0.889), consistent with count-based growth rather than a fixed additive effect.
>
> **What this teaches:** `Profile Visits` and `Likes` are both statistically significant predictors of new follows (p<0.001), while `Saves` is not — a person saving a post to revisit later doesn't reliably translate into a new follower the way a profile visit does. The Poisson GLM's 0.746 pseudo-R² also shows count-based regression can fit this kind of engagement data well without forcing it through a continuous-outcome model that doesn't match its shape.
>
> See the underlying theory on the Generalized Linear Models → page.

> **4. Click-Through Rate Prediction — Support Vector Machines**
>
> 10,000 ad impressions with site-engagement and demographic features, perfectly balanced click/no-click target. **Promoted to a full executed project** — SVM (72.0% accuracy, 0.7728 AUC) vs. Logistic Regression baseline (71.0%, 0.7744 AUC).
>
> See the full project →

> **5. Credit Score Classification — Kernel Methods Beyond SVM**
>
> 100,000 genuinely messy bank-customer records (corrupted ages, string-encoded numbers) classified into Good/Standard/Poor credit tiers. **Promoted to a full executed project** — Random Forest reaches 77.5% accuracy, 0.76 macro F1.
>
> See the full project →

> **6. Customer Segmentation — Hierarchical Clustering & DBSCAN**
>
> 8,950 credit card holders across 18 usage-behaviour features. **Promoted to a full executed project** — K-Means with k=3 (silhouette 0.251) reveals Big Spenders, Low-Engagement, and Cash-Advance-Reliant segments.
>
> See the full project →

> **Dataset note**
>
> ### 7. Fashion Recommendations — Recommender Systems
>
> A 44,424-product fashion catalogue (gender, category, article type, colour, season, usage occasion) — the same style of attribute set a Flipkart or Myntra-style fashion storefront would hold. A content-based recommender suggests visually/stylistically similar items using nothing but these categorical attributes — no images, no purchase history needed.
>
> The 44,424-item catalogue is dominated by Apparel and Accessories — the Topwear subset used for this demo alone contains over 15,000 shirts, t-shirts, and tops.
>
> **What this teaches:** content-based filtering doesn't need deep learning or image embeddings to be useful — one-hot encoding a handful of well-chosen categorical attributes and measuring cosine similarity between products already recovers meaningful "shop the look" recommendations. The similarity score of exactly 1.000 for several items also exposes a real limitation: once every one-hot attribute matches, the encoding can't distinguish "near-identical" from "actually the same shirt from a different brand" — a genuine argument for adding continuous features (price, popularity) to break such ties in a production system.
>
> > statso.io's original Fashion & Color Recommendation dataset (Kaggle) could not be mirrored outside Kaggle's own hosting. This treatment instead uses a comparable, verified, publicly-mirrored fashion product catalogue (Param Aggarwal's Fashion Product Images metadata, 44,424 rows) that supports the identical content-based filtering task with real attributes and real output.
>
> See the underlying theory on the Recommender Systems → page.

> **A genuine embedding-quality pitfall**
>
> ### 8. News Recommendation — K-Nearest Neighbours
>
> 180 Indian-news headlines, each already reduced to a 100-dimensional word2vec embedding vector plus a category label (Others 60, Entertainment 48, Politics 32, Sports 14, Business 12, Crime 11, Education 3). Since the embeddings are pre-computed, recommending similar articles becomes a plain KNN nearest-neighbour search over the vectors — no text processing needed.
>
> > Every pairwise cosine similarity in this dataset sits between 0.9997 and 1.0000 — the averaged word2vec vectors have **directionally collapsed**, meaning "which article is most similar" by raw ranking is nearly meaningless (everything looks equally similar). And yet **cosine-metric KNN classification** still recovers real category structure at 62.8% accuracy — nearly double the 33.3% majority baseline — while Euclidean KNN does *worse than guessing* at 25.0%. The lesson: a distance metric being appropriate for classification and a similarity score being meaningful for ranking are two different questions, and this embedding answers them differently.
>
> 3-fold cross-validated KNN accuracy: cosine distance nearly doubles the majority baseline, while Euclidean distance on the same collapsed vectors performs worse than always guessing "Others."
>
> **What this teaches:** don't assume an embedding is "good" just because a downstream classifier works reasonably well on it — always sanity-check the raw similarity distribution first. Here, a recommendation engine built naively on cosine-similarity *ranking* would silently return near-random results, even though the same vectors support a perfectly serviceable KNN classifier when the right distance metric is chosen.
>
> See the underlying theory on the K-Nearest Neighbours → page.

> **9. Instagram Recommendations — PCA**
>
> The same 119-post Instagram engagement dataset as Case Study 3, used for a different purpose here: reducing 11 numeric engagement metrics (impressions, reach sources, saves, comments, shares, likes, profile visits, follows) to a handful of principal components, so posts with a similar underlying engagement "shape" can be grouped for content recommendation.
>
> Scree plot: the first principal component alone captures 59.3% of variance across all 11 engagement metrics, and just 4 components reach 91.2% — a 11-to-4 dimensionality reduction with almost no information loss.
>
> **What this teaches:** PC1 (59.3% of variance) is almost certainly a general "overall engagement volume" axis — posts that get more impressions tend to get more of everything else too. PC2 and PC3 together add another 24.7%, likely separating posts that spread through hashtags/explore from posts that spread through direct-follower reach. Once posts are represented in this compressed 3-4 dimensional space instead of the original 11 raw counts, finding "posts with a similar engagement shape" becomes a simple nearest-neighbour search that isn't dominated by whichever raw metric happens to have the largest numbers.
>
> See the underlying theory on the Principal Component Analysis → page.

> **10. Screen Time Analysis — Bootstrap Resampling & Confidence Intervals**
>
> 54 days of personal screen-time logs (27 days each for Instagram and WhatsApp) — daily minutes used, notifications received, and times opened. With only 27 observations per app, assuming a normal distribution for a confidence interval is shaky; bootstrap resampling sidesteps that assumption entirely.
>
> 10,000-resample bootstrap distributions of the mean for each app, with the 95% percentile confidence interval marked. The two intervals don't overlap at all — a visual confirmation that WhatsApp usage is genuinely, not just coincidentally, higher.
>
> **What this teaches:** the two apps' 95% confidence intervals — [20.3, 43.1] for Instagram and [77.5, 121.6] for WhatsApp — don't overlap at all, so this small sample is enough to say confidently that average WhatsApp usage genuinely exceeds Instagram usage for this user, without ever needing to assume the underlying daily-usage distribution is normal. Bootstrapping is exactly the right tool when a sample is small and you don't want to bet the analysis on a distributional assumption you can't verify.
>
> See the underlying theory on the Bootstrap Resampling → page.

> **11. A/B Testing Analysis — Model Evaluation (Statistical Significance)**
>
> 294,478 website visitors split into a landing-page A/B test. **Promoted to a full executed project** — after cleaning 3,893 mismatched assignment rows, a two-proportion z-test finds *no* statistically significant difference (p=0.19) between old and new page conversion rates.
>
> See the full project →

> **12. Handling Missing Values Using Linear Regression — Data Preprocessing**
>
> 414 Taiwan real-estate transactions (transaction date, house age, distance to nearest MRT station, nearby convenience stores, latitude/longitude, price per unit area). The source data is clean — no missing values — so this treatment injects 20% missing values into the price column itself, then compares two ways of filling them back in. Kept as its genuine Taiwan dataset — not reframed with Indian branding.
>
> Regression imputation cuts imputation error by roughly 38% versus a flat median fill, by actually using each property's own age, MRT-distance, and location instead of ignoring them.
>
> **What this teaches:** median-filling assumes every missing value is close to the "typical" one — it ignores everything else you know about that specific row. Regression imputation uses a model trained on the observed rows (here reaching R²≈0.58 on those same rows) to make a property-specific estimate instead. The lesson generalizes well beyond real estate: whenever missing values correlate with other available features, regression is often a stronger imputation choice than a fixed statistic — using a predictive model as a preprocessing tool, not just a final-answer tool.
>
> See related preprocessing techniques on the Data Preparation → page.

> **13. Feature Selection — Feature Engineering**
>
> 1,000 real Ola/Uber-style ride-sharing bookings, dynamically priced with rider/driver counts, loyalty status, time of booking, vehicle type, and expected ride duration. Real correlation checks, not assumptions, decide which features actually matter for predicting cost.
>
> Correlation of each candidate feature with ride cost. Expected ride duration dominates almost completely (r=0.928); every other feature checked barely correlates at all (|r|<0.04).
>
> **What this teaches:** it would be easy to assume that a busy market (many riders, few drivers) drives up dynamic-pricing cost — that's the intuitive story behind surge pricing. The data says otherwise for this dataset: ride duration alone carries almost all the predictive signal, while rider count, driver count, past-ride count, and rating are all statistically negligible. Feature selection isn't about including everything that sounds plausible — it's about checking, and this genuine result would change which features a model-builder bothers to engineer further.
>
> See the underlying theory on the Feature Engineering → page.

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Which model has the best ROC-AUC?

For each fitted model in `models`, compute the test ROC-AUC (`roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])`). Store the dict in `aucs` and the best model's name in `best_model`.

In [ ]:
from sklearn.metrics import roc_auc_score
aucs = {}
best_model = None   # TODO


In [ ]:
try:
    check("six models", len(aucs) == 6)
    check("best is the max", best_model == max(aucs, key=aucs.get))
    check("logistic regression wins, as in the lesson", best_model == "Logistic Regression")
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.metrics import roc_auc_score
aucs = {n: roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]) for n, m in models.items()}
best_model = max(aucs, key=aucs.get)

```

</details>

### Exercise 2 · Medium · Prioritise recall on the malignant class

`y = 1` means benign and `0` means malignant. Compute the recall **for the malignant class** (`pos_label=0`) of every model and store the name of the best one in `best_malignant_recall`, with the dict in `mal_recall`.

In [ ]:
mal_recall = {}
best_malignant_recall = None   # TODO


In [ ]:
try:
    check("six models", len(mal_recall) == 6)
    check("best is the max", mal_recall[best_malignant_recall] == max(mal_recall.values()))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
mal_recall = {n: recall_score(y_te, m.predict(X_te), pos_label=0) for n, m in models.items()}
best_malignant_recall = max(mal_recall, key=mal_recall.get)

```

</details>

### Exercise 3 · Stretch · Count the missed tumours

For the random forest, use `confusion_matrix(y_te, pred)` (labels 0 = malignant, 1 = benign) to store the number of **malignant tumours predicted benign** (the dangerous error) in `missed`.

In [ ]:
from sklearn.metrics import confusion_matrix
missed = None   # TODO


In [ ]:
try:
    cm = confusion_matrix(y_te, models["Random Forest"].predict(X_te))
    check("count", missed == cm[0, 1])
    check("a small number", 0 <= missed <= 10)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_te, models["Random Forest"].predict(X_te))
missed = int(cm[0, 1])

```

In diagnosis the two errors are not symmetric: a missed malignancy (false negative) is far worse than a false alarm.

</details>

---
*Back to the course: **Machine Learning End To End → Case Studies Hub**.*